# Integration Assessment

Evaluate batch integration quality after scVI (and optionally Harmony).

**Metrics:**
- UMAP colored by batch, tissue type, HIV status
- Per-cluster batch composition heatmap
- scib metrics: batch ASW, graph connectivity, cell type ASW
- Comparison of scVI vs Harmony (if both were run)

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils import load_config, set_plotting_defaults
set_plotting_defaults()

cfg = load_config('../pipeline/config.yaml')

arm = 'primary'  # Change to 'focused' for sorted samples

In [ ]:
# Load integrated data
adata = sc.read_h5ad(str(Path(cfg['paths']['integration_output']) / f'integrated_{arm}.h5ad'))
print(f"Integrated data: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Batches: {adata.obs['Batch'].nunique()}")
print(f"Samples: {adata.obs['sample_id'].nunique()}")
print(f"Patients: {adata.obs['patientID'].nunique()}")

In [ ]:
# UMAP by batch, tissue type, HIV status, and cluster
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

sc.pl.umap(adata, color='Batch', ax=axes[0, 0], show=False, title='Batch')
sc.pl.umap(adata, color='tissueType', ax=axes[0, 1], show=False, title='Tissue type')
sc.pl.umap(adata, color='HIVstatus', ax=axes[1, 0], show=False, title='HIV status')
sc.pl.umap(adata, color='leiden_0.8', ax=axes[1, 1], show=False, title='Leiden (res=0.8)')

plt.tight_layout()
plt.show()

In [ ]:
# Per-cluster batch composition heatmap
batch_comp = pd.crosstab(adata.obs['leiden_0.8'], adata.obs['Batch'], normalize='index')

fig, ax = plt.subplots(figsize=(12, max(6, batch_comp.shape[0] * 0.4)))
sns.heatmap(batch_comp, cmap='YlOrRd', annot=True, fmt='.2f', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Fraction'})
ax.set_title('Batch composition per cluster (row-normalized)')
ax.set_xlabel('Batch')
ax.set_ylabel('Leiden cluster')
plt.tight_layout()
plt.show()

# Flag clusters dominated by a single batch
max_batch_frac = batch_comp.max(axis=1)
dominant = max_batch_frac[max_batch_frac > 0.7]
if len(dominant) > 0:
    print("WARNING: Clusters with >70% from a single batch (possible batch artifacts):")
    for cl, frac in dominant.items():
        batch = batch_comp.loc[cl].idxmax()
        print(f"  Cluster {cl}: {frac:.0%} from batch {batch}")

In [ ]:
# Per-cluster HIV status composition
hiv_comp = pd.crosstab(adata.obs['leiden_0.8'], adata.obs['HIVstatus'], normalize='index')

fig, ax = plt.subplots(figsize=(6, max(6, hiv_comp.shape[0] * 0.4)))
hiv_comp.plot(kind='barh', stacked=True, ax=ax, color=['steelblue', 'firebrick'])
ax.set_xlabel('Fraction')
ax.set_title('HIV status per cluster')
ax.legend(title='HIV status')
plt.tight_layout()
plt.show()

In [ ]:
# scib integration metrics (if available)
metrics_path = Path(cfg['paths']['integration_output']) / f'integration_metrics_{arm}.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print("Integration quality metrics:")
    print(metrics.T.to_string())
else:
    print("No scib metrics found. Install scib and re-run integration.")

In [ ]:
# Compare multiple clustering resolutions
res_cols = [c for c in adata.obs.columns if c.startswith('leiden_')]
if res_cols:
    n_cols = min(len(res_cols), 4)
    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]
    for ax, col in zip(axes, res_cols[:n_cols]):
        sc.pl.umap(adata, color=col, ax=ax, show=False,
                   title=f"{col} ({adata.obs[col].nunique()} clusters)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Sample distribution summary
sample_counts = adata.obs.groupby(['sample_id', 'Batch', 'HIVstatus', 'tissueType']).size().reset_index(name='n_cells')
print(f"Cells per sample (min={sample_counts['n_cells'].min()}, "
      f"median={sample_counts['n_cells'].median():.0f}, "
      f"max={sample_counts['n_cells'].max()})")
print(f"\nHIV status: {adata.obs['HIVstatus'].value_counts().to_dict()}")
print(f"Tissue type: {adata.obs['tissueType'].value_counts().to_dict()}")
print(f"\nSample counts:")
sample_counts.sort_values('n_cells')